In [ ]:
from pathlib import Path
import pandas as pd

# Direktori utama proyek
PROJECT_DIR = Path.cwd().parent

# Direktori eksperimen LODO v2
LODO_DIR = PROJECT_DIR / "datasets" / "experiments" / "v2"

# Daftar skenario
SCENARIOS = [
    "lodo_1_pabrik",
    "lodo_2_kalteng",
    "lodo_3_malaysia",
]

# Pastikan direktori tersedia
assert LODO_DIR.exists(), f"Folder tidak ditemukan: {LODO_DIR}"

for scenario in SCENARIOS:
    scenario_dir = LODO_DIR / scenario
    assert scenario_dir.exists(), f"Skenario tidak ditemukan: {scenario_dir}"

print("Konfigurasi berhasil.")
print("Project:", PROJECT_DIR)
print("LODO v2:", LODO_DIR)
print("Skenario:", SCENARIOS)


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def validate_image_label_pairs(split_dir):
    image_dir = split_dir / "images"
    label_dir = split_dir / "labels"

    images = {
        path.stem
        for path in image_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    }

    labels = {
        path.stem
        for path in label_dir.glob("*.txt")
        if path.is_file()
    }

    missing_labels = sorted(images - labels)
    orphan_labels = sorted(labels - images)

    return {
        "images": len(images),
        "labels": len(labels),
        "missing_labels": len(missing_labels),
        "orphan_labels": len(orphan_labels),
        "missing_examples": missing_labels[:5],
        "orphan_examples": orphan_labels[:5],
    }


results = []

for scenario in SCENARIOS:
    for split in ["train", "val", "test"]:
        split_dir = LODO_DIR / scenario / split

        result = validate_image_label_pairs(split_dir)

        results.append({
            "scenario": scenario,
            "split": split,
            **result
        })

pair_df = pd.DataFrame(results)

display(pair_df)

In [ ]:
def validate_annotations(label_dir):
    total_boxes = 0
    invalid_format = 0
    invalid_class = 0
    invalid_coordinates = 0
    invalid_dimensions = 0
    empty_labels = 0

    class_counts = {0: 0, 1: 0}

    for label_path in label_dir.glob("*.txt"):
        lines = label_path.read_text(encoding="utf-8").splitlines()

        if not lines:
            empty_labels += 1
            continue

        for line in lines:
            parts = line.split()

            if len(parts) != 5:
                invalid_format += 1
                continue

            try:
                class_id = int(parts[0])
                x_center, y_center, width, height = map(float, parts[1:])
            except ValueError:
                invalid_format += 1
                continue

            total_boxes += 1

            if class_id not in [0, 1]:
                invalid_class += 1
            else:
                class_counts[class_id] += 1

            coordinates = [x_center, y_center, width, height]

            if not all(0 <= value <= 1 for value in coordinates):
                invalid_coordinates += 1

            if width <= 0 or height <= 0:
                invalid_dimensions += 1

    return {
        "total_boxes": total_boxes,
        "unripe_id_0": class_counts[0],
        "ripe_id_1": class_counts[1],
        "empty_labels": empty_labels,
        "invalid_format": invalid_format,
        "invalid_class": invalid_class,
        "invalid_coordinates": invalid_coordinates,
        "invalid_dimensions": invalid_dimensions,
    }


annotation_results = []

for scenario in SCENARIOS:
    for split in ["train", "val", "test"]:
        label_dir = LODO_DIR / scenario / split / "labels"

        result = validate_annotations(label_dir)

        annotation_results.append({
            "scenario": scenario,
            "split": split,
            **result
        })

annotation_df = pd.DataFrame(annotation_results)

display(annotation_df)

In [ ]:
def get_image_stems(image_dir):
    return {
        path.stem
        for path in image_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    }


leakage_results = []

for scenario in SCENARIOS:
    scenario_dir = LODO_DIR / scenario

    train_images = get_image_stems(scenario_dir / "train" / "images")
    val_images = get_image_stems(scenario_dir / "val" / "images")
    test_images = get_image_stems(scenario_dir / "test" / "images")

    train_val = train_images & val_images
    train_test = train_images & test_images
    val_test = val_images & test_images

    leakage_results.append({
        "scenario": scenario,
        "train_val_overlap": len(train_val),
        "train_test_overlap": len(train_test),
        "val_test_overlap": len(val_test),
        "total_overlap": len(train_val) + len(train_test) + len(val_test),
    })

leakage_df = pd.DataFrame(leakage_results)

display(leakage_df)

In [ ]:
import hashlib

def calculate_image_hash(image_path):
    sha256 = hashlib.sha256()

    with open(image_path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


def get_image_hashes(image_dir):
    hashes = {}

    for image_path in image_dir.iterdir():
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            hashes[image_path.name] = calculate_image_hash(image_path)

    return hashes


hash_leakage_results = []

for scenario in SCENARIOS:
    scenario_dir = LODO_DIR / scenario

    split_hashes = {
        split: set(get_image_hashes(scenario_dir / split / "images").values())
        for split in ["train", "val", "test"]
    }

    train_val = split_hashes["train"] & split_hashes["val"]
    train_test = split_hashes["train"] & split_hashes["test"]
    val_test = split_hashes["val"] & split_hashes["test"]

    hash_leakage_results.append({
        "scenario": scenario,
        "train_val_identical": len(train_val),
        "train_test_identical": len(train_test),
        "val_test_identical": len(val_test),
        "total_identical": len(train_val) + len(train_test) + len(val_test),
    })

hash_leakage_df = pd.DataFrame(hash_leakage_results)

display(hash_leakage_df)

In [ ]:
VALIDATION_DIR = PROJECT_DIR / "results" / "validation_lodo_v2"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

pair_df.to_csv(
    VALIDATION_DIR / "image_label_validation.csv",
    index=False
)

annotation_df.to_csv(
    VALIDATION_DIR / "annotation_validation.csv",
    index=False
)

leakage_df.to_csv(
    VALIDATION_DIR / "filename_leakage_validation.csv",
    index=False
)

hash_leakage_df.to_csv(
    VALIDATION_DIR / "sha256_leakage_validation.csv",
    index=False
)

print("Semua laporan validasi berhasil disimpan.")
print("Lokasi:", VALIDATION_DIR)